# Tutorial 1: mBuild Polymers
### Learning Objectives
* Understand what SMILES strings are and how they represent molecular structures
* Set up mBuild and import necessary modules
* Understand the Polymer class and its basic components
### Key Points
* SMILES (Simplified Molecular Input Line Entry System) is a notation for representing molecular structures
* mBuild uses a hierarchical Compound object to represent molecular structures
* The Polymer class connects monomer units together in specified sequences
* Tagged SMILES allow us to mark bonding sites on molecules

## Setup
------

In [ ]:
# Import necessary libraries
import mbuild as mb
from mbuild.polymer import Polymer

# Check mBuild version
print(f"mBuild version: {mb.__version__}")

## What is SMILES?
-----
SMILES is a compact notation for molecular structures:

* C = carbon</br>
* CC = ethane (two carbons bonded)</br>
* c1ccccc1 = benzene ring</br>
* [C{tag}] = tagged carbon (used for bonding sites)</br>

*Tagged SMILES* extend SMILES by marking specific atoms where polymers will bond.</br>


In [ ]:
# Create a tagged monomer

# Define a simple tagged SMILES for ethane with marked bonding sites
ethane_smiles = "C{head}C{tail}"

# Create monomer from SMILES
# (This uses mBuild's SMILES import functionality via RDKit)
monomer = mb.load(ethane_smiles, smiles=True)

print(f"Monomer created: {type(monomer)}")
print(f"Number of particles: {len(list(monomer.particles()))}")
for part, tag in monomer.tags:
    print(f"{part.name} | {part.n_direct_bonds} | {tag}")

<div style="background-color:rgb(0, 0, 0); border-left: 5px solidrgb(137, 196, 241); padding: 10px;">
  <strong>Note:</strong> If you are familiar with BigSMILES (10.1021/acscentsci.9b00476), it is possible to utilize that notation in you tagging. However, we generally recommend tail and head tagging for it's general readability.
</div>

## Creating a Linear Polymer
--------

In [ ]:
# Create a Polymer instance
polymer = Polymer()

# Add a monomer using the tagged SMILES
# head_tag and tail_tag mark where bonds form
polymer.add_monomer(
    compound=monomer,
    head_tag="head",
    tail_tag="tail",
    separation=0.15,  # bond length in nm
    bond_order=1
)

print(f"Monomers types in polymer: {len(polymer.monomers)}")

# Build 5 repeat units
polymer.build(n=5, sequence="A")

print(f"Total particles in polymer: {len(list(polymer.particles()))}")
print(f"Polymer built successfully!")

## Creating a Block Copolymer
-----

In [ ]:
# Create a Polymer instance
copolymer = Polymer()

# Create two block copolymers
a_block = mb.load("C{head}C{tail}C1=CC=CC=CC1", smiles=True)
# a_block = mb.load("C{head}{tail}1CCCCC1", smiles=True) # TODO: failing!!
b_block = mb.load("C{head}C=CC{tail}", smiles=True)

# Add a monomer using the tagged SMILES
# head_tag and tail_tag mark where bonds form
copolymer.add_monomer(
    compound=a_block,
    head_tag="head",
    tail_tag="tail",
    separation=0.15,  # bond length in nm
    bond_order=1
)
copolymer.add_monomer(
    compound=b_block,
    head_tag="head",
    tail_tag="tail",
    separation=0.15,  # bond length in nm
    bond_order=1
)

print(f"Monomers in polymer: {len(copolymer.monomers)}")

# Build 5 repeat units
copolymer.build(n=20, sequence="AB")

print(f"Total particles in polymer: {len(list(copolymer.particles()))}")
print(f"Polymer built successfully!")
copolymer.visualize()

[!NOTE]

Here are some example monomer systems that could be used, with `h1` and `t1` as the `head` and `tail` tags

| SMILES | Description | Monomer Type | Common Use |
|--------|-------------|--------------|------------|
| `C{h1}{t1}` | Methylene | Aliphatic | Polyethylene backbone |
| `C{h1}CCC{t1}` | Butylene | Aliphatic | Longer alkyl backbone |
| `C{h1}{t1}1CCCCC1` | Cyclohexane | Cycloaliphatic | Rigid cyclic units |
| `c{h1}1cc{t1}ccc1` | Phenyl | Aromatic | Aromatic polymers, polyesters |
| `c{h1}C{t1}1ccncc1` | Pyridine | Aromatic/N-heterocycle | Poly(vinylpyridine) |
| `C{h1}c1cc{t1}ccc1O` | Phenol | Aromatic/Hydroxyl | Phenolic resins, bio-inspired |
| `C{h1}C(C)C` | Isobutyl | Branched | Semi-Branched chain architectures |
| `C{h1}=C` | Octene | Unsaturated | Conjugated polymers |
| `C{h1}#N` | Nitrile | Functional | Cyano-functional polymers |
| `C{h1}C#CC` | Alkyne | Unsaturated | Acetylenic polymers |
| `[Fe+2]{h1}` | Iron(II) | Metal | Coordination polymers, metallopolymers |
| `[Fe+3]{h1}` | Iron(III) | Metal | Coordination polymers, metallopolymers |
| `[Mg+2]{h1}` | Magnesium(II) | Metal | Metal-organic frameworks |
| `[Na+]{h1}` | Sodium | Metal ion | Ionic polymers, salt-bridges |
| `[K+]{h1}` | Potassium | Metal ion | Ionic polymers |
| `[NH2]{h1}` | Amino | Functional | Bio-inspired, peptide-like |
| `[OH]{h1}` | Hydroxyl | Functional | Polyols, bio-polymers |
| `[O-]{h1}` | Carboxylate | Anionic | Polyelectrolytes, bio-inspired |
| `[CH3]C(=O)O{h1}` | Acetate | Ester | Polymer side-chains |
| `c{h1}1cnc[nH]1` | Imidazole | N-heterocycle | Bio-inspired, metal coordination |
| `c{h1}1ccncc1O` | Hydroxypyridine | N-heterocycle/Hydroxyl | Bio-functional |
| `c{h1}1ccncc1C(=O)O` | Pyridinecarboxylic acid | Aromatic/Carboxylic acid | Bio-inspired, coordinating |

## BigSMILES Example
----------

In [ ]:
# BigSMILES Use < <,  > > tags to denote directionality.
# NOTE: `<` bonds to `>``. 
# NOTE: Tags are always placed to the left of their associated atom. The exception is if the tag
# corresponds to the first atom, then it can placed at the beginning of the string.
big_EVA_smiles = "{<CC<,>CC(OC(=O)C)>}"
# big_EVA_smiles = "{$CC$,$CC(OC(=O)C)$}" # also works
polymer = Polymer.from_big_smiles(big_EVA_smiles, bond_separation=0.8)

# build with A and B from the comma seperated values in the big smiles
polymer.build(n=10, sequence="ABA")

from mbuild.simulation import energy_minimize
energy_minimize(polymer)
polymer.visualize(show_ports=True)


## Custom EndGroups
-----

Self-immolative molecules (10.1021/jacs.1c11410) with programmed terminal groups.
</br>TODO: Image from paper

In [ ]:
# Create a Polymer instance
copolymer = Polymer()

# Create two block copolymers
quinone = mb.load("O{head}C(=O)NC1=CC=C(C{tail})C=CC1", smiles=True)

# Add a monomer using the tagged SMILES
# head_tag and tail_tag mark where bonds form
copolymer.add_monomer(
    compound=quinone,
    head_tag="head",
    tail_tag="tail",
    separation=0.15,  # bond length in nm
)
head = mb.load("N{head}C(=O)", smiles=True) # create amide head terminus
tail = mb.load("O{tail}", smiles=True) # create hydroxyl tail terminus
copolymer.add_end_groups(head, separation=0.15, bond_tag="head")
copolymer.add_end_groups(tail, separation=0.15, bond_tag="tail")

# Build 5 repeat units
copolymer.build(n=5)

copolymer.visualize()

### Exercise: 